# Fine-tune contrastive-pretrained ResNet-18, ps32 SCV

This notebook fine-tunes a binary ResNet-18 classifier initialized from the SimCLR-style contrastive SSL encoder checkpoint. It uses the same ps32 5-cluster spatial cross-validation protocol as the scratch baseline and masked-reconstruction fine-tuning experiment.


## 1. Purpose and experiment configuration

This stage performs supervised fine-tuning only. It does not run contrastive SSL pretraining, does not modify the contrastive checkpoint, and does not generate susceptibility maps. The methodological contrast against the scratch baseline is encoder initialization: random weights versus the contrastive SSL-pretrained encoder.


In [1]:
from pathlib import Path
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

model_name = "contrastive_resnet18"
pretraining = "contrastive"
patch_size = 32
patch_tag = "ps32"
random_seed = 42
val_fraction = 0.2
batch_size = 16
encoder_learning_rate = 1e-5
head_learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 100
early_stopping_patience = 15
gradient_clip_norm = 5.0
dropout = 0.4
num_workers = 0

scratch_mean_auc = 0.778842
scratch_std_auc = 0.104754
scratch_worst_fold_auc = 0.624023
masked_recon_mean_auc = 0.820912
masked_recon_std_auc = 0.043011
masked_recon_worst_fold_auc = 0.786325

patch_index_csv = PROJECT_ROOT / "data" / "processed" / "patches" / "labeled_patch_index_ps32_common_balanced.csv"
raster_dir = PROJECT_ROOT / "data" / "processed" / "rasters_cleaned"
encoder_checkpoint_path = PROJECT_ROOT / "checkpoints" / "ssl_pretrained" / "contrastive" / "resnet18_contrastive_ps32_encoder_best.pt"
output_root = PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "contrastive"
figure_root = PROJECT_ROOT / "figures" / "R1_ssl_task_comparison" / "contrastive"
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "finetuned" / "contrastive_resnet18"

prediction_dir = output_root / "predictions"
metrics_dir = output_root / "metrics"
training_log_dir = output_root / "training_logs"
roc_dir = figure_root / "roc_curves"
pr_dir = figure_root / "pr_curves"
training_curve_dir = figure_root / "training_curves"

fold_metrics_path = metrics_dir / "contrastive_resnet18_ps32_fold_metrics.csv"
summary_metrics_path = metrics_dir / "contrastive_resnet18_ps32_summary_metrics.csv"
roc_figure_path = roc_dir / "contrastive_resnet18_ps32_roc_all_folds.png"
pr_figure_path = pr_dir / "contrastive_resnet18_ps32_pr_all_folds.png"
training_curve_path = training_curve_dir / "contrastive_resnet18_ps32_training_curves.png"

print("Project root:", PROJECT_ROOT)
print("Patch index:", patch_index_csv)
print("Contrastive encoder checkpoint:", encoder_checkpoint_path)


Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Patch index: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\patches\labeled_patch_index_ps32_common_balanced.csv
Contrastive encoder checkpoint: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_encoder_best.pt


## 2. Import packages and project modules


In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from src.metrics import compute_binary_metrics, find_best_f1_threshold
from src.models_resnet18 import create_resnet18_binary_classifier, load_ssl_encoder_weights_into_classifier
from src.patch_dataset import RasterPatchDataset, list_raster_files, audit_raster_alignment
from src.plotting import (
    plot_pr_curves_all_folds,
    plot_roc_curves_all_folds,
    plot_training_curves,
    set_publication_plot_style,
)
from src.train_finetune import evaluate_model, train_scv_fold
from src.utils import count_trainable_parameters, ensure_dir, load_checkpoint, set_global_seed

for directory in [prediction_dir, metrics_dir, training_log_dir, roc_dir, pr_dir, training_curve_dir, checkpoint_dir]:
    ensure_dir(directory)


## 3. Set random seeds and device


In [3]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = bool(torch.cuda.is_available())
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    print("torch.version.cuda:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available; training will run on CPU.")


Using device: cuda
CUDA GPU: NVIDIA GeForce RTX 4090
torch.version.cuda: 11.8


## 4. Set publication-quality plotting style


In [4]:
plot_font_family = set_publication_plot_style(font_family="Arial", font_size=10)
print("Plotting font family:", plot_font_family)
print("Plotting font size:", 10)


Plotting font family: Arial
Plotting font size: 10


## 5. Load patch index, cleaned rasters, and contrastive pretrained encoder checkpoint


In [5]:
patch_index = pd.read_csv(patch_index_csv).reset_index(drop=True)
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=-9999)
encoder_checkpoint = torch.load(encoder_checkpoint_path, map_location="cpu")
expected_checkpoint_keys = {"encoder_state_dict", "epoch", "val_loss", "config", "channel_means", "channel_stds"}
missing_checkpoint_keys = expected_checkpoint_keys - set(encoder_checkpoint.keys())
if missing_checkpoint_keys:
    raise KeyError(f"Contrastive encoder checkpoint is missing keys: {sorted(missing_checkpoint_keys)}")
print("Patch index rows:", len(patch_index))
print("Number of raster channels:", len(raster_files))
print("Raster shape:", (raster_audit.height, raster_audit.width))
print("SSL checkpoint epoch:", encoder_checkpoint.get("epoch"))
print("SSL checkpoint val_loss:", encoder_checkpoint.get("val_loss"))
print("SSL encoder state keys:", len(encoder_checkpoint["encoder_state_dict"]))


Patch index rows: 344
Number of raster channels: 14
Raster shape: (10071, 9596)
SSL checkpoint epoch: 45
SSL checkpoint val_loss: 1.1136543502807617
SSL encoder state keys: 120


## 6. Verify input data quality


In [6]:
required_columns = ["sample_id", "x", "y", "label", "source", "cluster_id", "valid_patch", "nodata_ratio"]
missing_columns = [column for column in required_columns if column not in patch_index.columns]
if missing_columns:
    raise ValueError(f"Patch index missing required columns: {missing_columns}")
if len(patch_index) != 344:
    print(f"WARNING: expected 344 labeled samples, found {len(patch_index)}.")
if not patch_index["valid_patch"].astype(bool).all():
    raise ValueError("Patch index contains invalid patches.")
if not np.isclose(patch_index["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
    raise ValueError("Patch index contains nonzero nodata_ratio rows.")
label_counts = patch_index["label"].value_counts().sort_index()
cluster_label_table = pd.crosstab(patch_index["cluster_id"], patch_index["label"])
if set(label_counts.index.astype(int)) != {0, 1}:
    raise ValueError("Labels must be 0 and 1 only.")
if int(label_counts.get(0, 0)) != int(label_counts.get(1, 0)):
    raise ValueError("Input labeled dataset is not balanced by label.")
if sorted(patch_index["cluster_id"].unique().tolist()) != [0, 1, 2, 3, 4]:
    raise ValueError("cluster_id values must be 0, 1, 2, 3, 4.")

print("total sample count:", len(patch_index))
print("label counts:")
print(label_counts.to_string())
print("cluster_id x label table:")
print(cluster_label_table.to_string())
print("patch_size:", patch_size)
print("number of raster channels:", len(raster_files))
print("dataset path:", patch_index_csv)
print("contrastive encoder checkpoint path:", encoder_checkpoint_path)
print("pretrained weights loaded successfully: checkpoint readable")
print("device:", device)
print("batch size:", batch_size)
print("encoder learning rate:", encoder_learning_rate)
print("head learning rate:", head_learning_rate)
print("max epochs:", max_epochs)
print("early stopping patience:", early_stopping_patience)
print("plotting font family and font size:", plot_font_family, 10)


total sample count: 344
label counts:
label
0    172
1    172
cluster_id x label table:
label        0   1
cluster_id        
0           32  32
1           39  39
2           53  53
3           14  14
4           34  34
patch_size: 32
number of raster channels: 14
dataset path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\patches\labeled_patch_index_ps32_common_balanced.csv
contrastive encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_encoder_best.pt
pretrained weights loaded successfully: checkpoint readable
device: cuda
batch size: 16
encoder learning rate: 1e-05
head learning rate: 0.0001
max epochs: 100
early stopping patience: 15
plotting font family and font size: Arial 10


## 7. Define modified ResNet-18 classifier and load contrastive SSL encoder weights


In [7]:
preview_model = create_resnet18_binary_classifier(
    in_channels=14,
    dropout=dropout,
    small_patch_stem=True,
    pretrained=False,
)
preview_load_info = load_ssl_encoder_weights_into_classifier(
    preview_model,
    encoder_checkpoint_path,
    strict_encoder=True,
)
loaded_preview_keys = len(preview_load_info["loaded_keys"])
if loaded_preview_keys < 110:
    raise RuntimeError(f"Too few contrastive encoder keys loaded: {loaded_preview_keys}")
print("preview loaded pretrained encoder keys:", loaded_preview_keys)
print("model parameter count:", count_trainable_parameters(preview_model))
del preview_model


Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']
preview loaded pretrained encoder keys: 120
model parameter count: 11175681


## 8. Define SCV fold loop

Fold loop follows cluster_id 0..4 as held-out test clusters, matching the scratch and masked-reconstruction experiments.


## 9. Compute fold-specific channel normalization from training data only

For fair comparison with scratch and masked-reconstruction fine-tuning, supervised normalization is computed from training samples within each fold only. SSL pretraining means/stds stored in the checkpoint are not used for supervised fine-tuning.


In [8]:
def compute_channel_stats_for_indices(indices, batch_size_for_stats=32):
    stats_dataset = RasterPatchDataset(
        patch_index_csv=patch_index_csv,
        raster_dir=raster_dir,
        patch_size=patch_size,
        nodata_value=-9999,
        normalize=False,
        return_metadata=False,
        valid_only=True,
    )
    loader = DataLoader(
        Subset(stats_dataset, list(indices)),
        batch_size=batch_size_for_stats,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    )
    channel_sum = None
    channel_sumsq = None
    pixel_count = 0
    for X, _y in loader:
        X = X.float()
        if channel_sum is None:
            channel_sum = X.sum(dim=(0, 2, 3))
            channel_sumsq = (X ** 2).sum(dim=(0, 2, 3))
        else:
            channel_sum += X.sum(dim=(0, 2, 3))
            channel_sumsq += (X ** 2).sum(dim=(0, 2, 3))
        pixel_count += X.shape[0] * X.shape[2] * X.shape[3]
    stats_dataset.close()
    means = (channel_sum / pixel_count).numpy().astype("float32")
    variances = (channel_sumsq / pixel_count).numpy() - means**2
    stds = np.sqrt(np.maximum(variances, 1e-12)).astype("float32")
    return means, stds


def make_normalized_dataset(channel_means, channel_stds):
    return RasterPatchDataset(
        patch_index_csv=patch_index_csv,
        raster_dir=raster_dir,
        patch_size=patch_size,
        nodata_value=-9999,
        normalize=True,
        channel_means=channel_means,
        channel_stds=channel_stds,
        return_metadata=True,
        valid_only=True,
    )


def build_optimizer(model):
    encoder_params = []
    head_params = []
    for name, parameter in model.named_parameters():
        if name.startswith("fc."):
            head_params.append(parameter)
        else:
            encoder_params.append(parameter)
    return torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": encoder_learning_rate},
            {"params": head_params, "lr": head_learning_rate},
        ],
        weight_decay=weight_decay,
    )


## 10. Fine-tune fold-specific contrastive-pretrained ResNet-18 models


In [9]:
fold_predictions = []
training_logs = []
fold_metric_rows = []
loaded_keys_by_fold = {}

for fold_id in range(5):
    print("\n" + "=" * 72)
    print(f"Fold {fold_id}: held-out test cluster {fold_id}")
    test_mask = patch_index["cluster_id"].astype(int) == fold_id
    train_candidate_indices = patch_index.index[~test_mask].to_numpy()
    test_indices = patch_index.index[test_mask].to_numpy()
    train_candidate_labels = patch_index.loc[train_candidate_indices, "label"].to_numpy()

    train_indices, val_indices = train_test_split(
        train_candidate_indices,
        test_size=val_fraction,
        random_state=random_seed + fold_id,
        stratify=train_candidate_labels,
    )
    train_indices = np.asarray(train_indices, dtype=int)
    val_indices = np.asarray(val_indices, dtype=int)
    test_indices = np.asarray(test_indices, dtype=int)

    train_ids = set(patch_index.loc[train_indices, "sample_id"].astype(str))
    val_ids = set(patch_index.loc[val_indices, "sample_id"].astype(str))
    test_ids = set(patch_index.loc[test_indices, "sample_id"].astype(str))
    if train_ids & val_ids or train_ids & test_ids or val_ids & test_ids:
        raise ValueError(f"Fold {fold_id} has overlapping sample IDs across splits.")
    if fold_id in set(patch_index.loc[train_indices, "cluster_id"].astype(int)):
        raise ValueError(f"Fold {fold_id}: test cluster leaked into training split.")
    if fold_id in set(patch_index.loc[val_indices, "cluster_id"].astype(int)):
        raise ValueError(f"Fold {fold_id}: test cluster leaked into validation split.")
    for split_name, split_indices in [("train", train_indices), ("val", val_indices), ("test", test_indices)]:
        split = patch_index.loc[split_indices]
        if not split["valid_patch"].astype(bool).all():
            raise ValueError(f"Fold {fold_id} {split_name} contains invalid patches.")
        if not np.isclose(split["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
            raise ValueError(f"Fold {fold_id} {split_name} contains nonzero nodata_ratio.")
        if split["label"].nunique() < 2:
            print(f"WARNING: Fold {fold_id} {split_name} split has only one class.")

    print("train clusters:", sorted(patch_index.loc[train_indices, "cluster_id"].unique().tolist()))
    print("validation size:", len(val_indices))
    print("test cluster:", fold_id)
    print("train label counts:")
    print(patch_index.loc[train_indices, "label"].value_counts().sort_index().to_string())
    print("val label counts:")
    print(patch_index.loc[val_indices, "label"].value_counts().sort_index().to_string())
    print("test label counts:")
    print(patch_index.loc[test_indices, "label"].value_counts().sort_index().to_string())

    channel_means, channel_stds = compute_channel_stats_for_indices(train_indices)
    print("channel means and stds shape:", channel_means.shape, channel_stds.shape)

    fold_dataset = make_normalized_dataset(channel_means, channel_stds)
    train_loader = DataLoader(
        Subset(fold_dataset, train_indices.tolist()),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        Subset(fold_dataset, val_indices.tolist()),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        Subset(fold_dataset, test_indices.tolist()),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    model = create_resnet18_binary_classifier(
        in_channels=14,
        dropout=dropout,
        small_patch_stem=True,
        pretrained=False,
    )
    load_info = load_ssl_encoder_weights_into_classifier(model, encoder_checkpoint_path, strict_encoder=True)
    loaded_keys_by_fold[fold_id] = len(load_info["loaded_keys"])
    if loaded_keys_by_fold[fold_id] < 110:
        raise RuntimeError(f"Fold {fold_id}: too few contrastive encoder keys loaded.")
    model = model.to(device)
    print("model parameter count:", sum(p.numel() for p in model.parameters()))
    print("number of trainable parameters:", count_trainable_parameters(model))
    print("number of loaded pretrained encoder keys:", loaded_keys_by_fold[fold_id])

    smoke_batch = next(iter(train_loader))
    smoke_X = smoke_batch[0].to(device, non_blocking=True)
    smoke_y = smoke_batch[1].to(device, non_blocking=True)
    print("device sanity model:", next(model.parameters()).device)
    print("device sanity X:", smoke_X.device)
    print("device sanity y:", smoke_y.device)
    if torch.cuda.is_available():
        assert next(model.parameters()).is_cuda and smoke_X.is_cuda and smoke_y.is_cuda

    optimizer = build_optimizer(model)
    criterion = nn.BCEWithLogitsLoss()
    checkpoint_path = checkpoint_dir / f"{model_name}_{patch_tag}_fold{fold_id}_best.pt"
    checkpoint_extra = {
        "fold": fold_id,
        "patch_size": patch_size,
        "model_name": model_name,
        "pretraining_type": pretraining,
        "encoder_checkpoint_path": str(encoder_checkpoint_path),
        "channel_means": channel_means,
        "channel_stds": channel_stds,
        "config": {
            "batch_size": batch_size,
            "encoder_learning_rate": encoder_learning_rate,
            "head_learning_rate": head_learning_rate,
            "weight_decay": weight_decay,
            "max_epochs": max_epochs,
            "early_stopping_patience": early_stopping_patience,
            "gradient_clip_norm": gradient_clip_norm,
            "dropout": dropout,
            "random_seed": random_seed,
            "val_fraction": val_fraction,
        },
    }

    training_log, best_payload = train_scv_fold(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        checkpoint_path=checkpoint_path,
        max_epochs=max_epochs,
        early_stopping_patience=early_stopping_patience,
        monitor_metric="val_auc",
        grad_clip_norm=gradient_clip_norm,
        checkpoint_extra=checkpoint_extra,
    )
    training_log["fold"] = fold_id
    training_log["encoder_learning_rate"] = encoder_learning_rate
    training_log["head_learning_rate"] = head_learning_rate
    log_path = training_log_dir / f"{model_name}_{patch_tag}_fold{fold_id}_training_log.csv"
    training_log.to_csv(log_path, index=False)
    training_logs.append(training_log)

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint.update(
        {
            "best_val_auc": best_payload.get("best_val_auc"),
            "best_val_loss": best_payload.get("best_val_loss"),
            "fold": fold_id,
            "patch_size": patch_size,
            "model_name": model_name,
            "pretraining_type": pretraining,
            "encoder_checkpoint_path": str(encoder_checkpoint_path),
            "channel_means": channel_means,
            "channel_stds": channel_stds,
            "config": checkpoint_extra["config"],
        }
    )
    torch.save(checkpoint, checkpoint_path)

    load_checkpoint(checkpoint_path, model, optimizer=None, map_location=device)
    val_result = evaluate_model(model, val_loader, criterion, device)
    best_threshold_f1, _ = find_best_f1_threshold(val_result["y_true"], val_result["y_probs"])
    test_result = evaluate_model(model, test_loader, criterion, device)
    test_metrics_05 = compute_binary_metrics(test_result["y_true"], test_result["y_probs"], threshold=0.5)
    test_metrics_best = compute_binary_metrics(test_result["y_true"], test_result["y_probs"], threshold=best_threshold_f1)

    test_meta = patch_index.loc[test_indices, ["sample_id", "x", "y", "label", "source", "cluster_id"]].reset_index(drop=True)
    pred_df = test_meta.copy()
    pred_df["fold"] = fold_id
    pred_df["y_true"] = test_result["y_true"].astype(int)
    pred_df["y_logit"] = test_result["y_logits"].astype(float)
    pred_df["y_prob"] = test_result["y_probs"].astype(float)
    pred_df["y_pred_05"] = (test_result["y_probs"] >= 0.5).astype(int)
    pred_df["y_pred_best_f1"] = (test_result["y_probs"] >= best_threshold_f1).astype(int)
    pred_df["split"] = "test"
    pred_df = pred_df[
        [
            "sample_id", "x", "y", "label", "source", "cluster_id", "fold",
            "y_true", "y_logit", "y_prob", "y_pred_05", "y_pred_best_f1", "split",
        ]
    ]
    pred_path = prediction_dir / f"{model_name}_{patch_tag}_fold{fold_id}_predictions.csv"
    pred_df.to_csv(pred_path, index=False)
    fold_predictions.append(pred_df)

    fold_metric_rows.append(
        {
            "model": model_name,
            "pretraining": pretraining,
            "patch_size": patch_size,
            "fold": fold_id,
            "n_train": len(train_indices),
            "n_val": len(val_indices),
            "n_test": len(test_indices),
            "n_test_pos": int((test_result["y_true"] == 1).sum()),
            "n_test_neg": int((test_result["y_true"] == 0).sum()),
            "auc": test_metrics_05["auc"],
            "pr_auc": test_metrics_05["pr_auc"],
            "accuracy_05": test_metrics_05["accuracy"],
            "precision_05": test_metrics_05["precision"],
            "recall_05": test_metrics_05["recall"],
            "f1_05": test_metrics_05["f1"],
            "best_threshold_f1": best_threshold_f1,
            "accuracy_best_f1": test_metrics_best["accuracy"],
            "precision_best_f1": test_metrics_best["precision"],
            "recall_best_f1": test_metrics_best["recall"],
            "f1_best_f1": test_metrics_best["f1"],
            "tn": test_metrics_best["tn"],
            "fp": test_metrics_best["fp"],
            "fn": test_metrics_best["fn"],
            "tp": test_metrics_best["tp"],
            "best_epoch": best_payload.get("best_epoch"),
            "best_val_auc": best_payload.get("best_val_auc"),
            "best_val_loss": best_payload.get("best_val_loss"),
        }
    )
    print(f"Fold {fold_id} test AUC: {test_metrics_05['auc']:.6f}; best threshold: {best_threshold_f1:.6f}")
    fold_dataset.close()

fold_metrics = pd.DataFrame(fold_metric_rows)
fold_metrics.to_csv(fold_metrics_path, index=False)
print("Saved fold metrics:", fold_metrics_path)



Fold 0: held-out test cluster 0
train clusters: [1, 2, 3, 4]
validation size: 56
test cluster: 0
train label counts:
label
0    112
1    112
val label counts:
label
0    28
1    28
test label counts:
label
0    32
1    32


channel means and stds shape: (14,) (14,)
Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']


model parameter count: 11175681
number of trainable parameters: 11175681
number of loaded pretrained encoder keys: 120


device sanity model: cuda:0
device sanity X: cuda:0
device sanity y: cuda:0


Fold 0 test AUC: 0.872070; best threshold: 0.357677

Fold 1: held-out test cluster 1
train clusters: [0, 2, 3, 4]
validation size: 54
test cluster: 1
train label counts:
label
0    106
1    106
val label counts:
label
0    27
1    27
test label counts:
label
0    39
1    39


...

device sanity model: cuda:0
device sanity X: cuda:0
device sanity y: cuda:0


Fold 3 test AUC: 0.806122; best threshold: 0.226218

Fold 4: held-out test cluster 4
train clusters: [0, 1, 2, 3]
validation size: 56
test cluster: 4
train label counts:
label
0    110
1    110
val label counts:
label
0    28
1    28
test label counts:
label
0    34
1    34


channel means and stds shape: (14,) (14,)
Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']
model parameter count: 11175681
number of trainable parameters: 11175681
number of loaded pretrained encoder keys: 120


device sanity model: cuda:0
device sanity X: cuda:0
device sanity y: cuda:0


Fold 4 test AUC: 0.897059; best threshold: 0.561240
Saved fold metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\contrastive\metrics\contrastive_resnet18_ps32_fold_metrics.csv


## 11. Evaluate fold-specific test performance


In [10]:
print(fold_metrics[["fold", "auc", "pr_auc", "accuracy_05", "f1_05", "best_threshold_f1", "f1_best_f1", "best_epoch"]].to_string(index=False))


 fold      auc   pr_auc  accuracy_05    f1_05  best_threshold_f1  f1_best_f1  best_epoch
    0 0.872070 0.819683     0.812500 0.818182           0.357677    0.805556          29
    1 0.704142 0.714570     0.705128 0.646154           0.553857    0.600000           4
    2 0.910644 0.883535     0.820755 0.842975           0.478679    0.815385           6
    3 0.806122 0.848101     0.714286 0.750000           0.226218    0.742857          35
    4 0.897059 0.889842     0.691176 0.571429           0.561240    0.409091          17


## 12. Save predictions, metrics, logs, and checkpoints


In [11]:
prediction_columns = [
    "sample_id", "x", "y", "label", "source", "cluster_id", "fold",
    "y_true", "y_logit", "y_prob", "y_pred_05", "y_pred_best_f1", "split",
]
metric_columns = [
    "model", "pretraining", "patch_size", "fold", "n_train", "n_val", "n_test",
    "n_test_pos", "n_test_neg", "auc", "pr_auc", "accuracy_05", "precision_05",
    "recall_05", "f1_05", "best_threshold_f1", "accuracy_best_f1",
    "precision_best_f1", "recall_best_f1", "f1_best_f1", "tn", "fp", "fn", "tp",
    "best_epoch", "best_val_auc", "best_val_loss",
]
log_columns = [
    "epoch", "train_loss", "val_loss", "train_auc", "val_auc", "train_f1", "val_f1",
    "learning_rate", "encoder_learning_rate", "head_learning_rate",
]
for fold_id in range(5):
    pred_path = prediction_dir / f"{model_name}_{patch_tag}_fold{fold_id}_predictions.csv"
    log_path = training_log_dir / f"{model_name}_{patch_tag}_fold{fold_id}_training_log.csv"
    ckpt_path = checkpoint_dir / f"{model_name}_{patch_tag}_fold{fold_id}_best.pt"
    pred = pd.read_csv(pred_path)
    log = pd.read_csv(log_path)
    if list(pred.columns) != prediction_columns:
        raise ValueError(f"Prediction columns mismatch for fold {fold_id}.")
    missing_log_cols = [column for column in log_columns if column not in log.columns]
    if missing_log_cols:
        raise ValueError(f"Training log fold {fold_id} missing columns: {missing_log_cols}")
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    for required_key in ["model_state_dict", "optimizer_state_dict", "epoch", "best_val_auc", "best_val_loss", "fold", "patch_size", "model_name", "pretraining_type", "encoder_checkpoint_path", "channel_means", "channel_stds", "config"]:
        if required_key not in ckpt:
            raise KeyError(f"Checkpoint fold {fold_id} missing key {required_key}.")
if list(fold_metrics.columns) != metric_columns:
    raise ValueError("Fold metrics columns do not match required schema.")
print("Prediction files, training logs, metrics, and checkpoints passed schema checks.")


Prediction files, training logs, metrics, and checkpoints passed schema checks.


## 13. Plot ROC, PR, and training curves using standardized figure style


In [12]:
plot_roc_curves_all_folds(fold_predictions, roc_figure_path)
plot_pr_curves_all_folds(fold_predictions, pr_figure_path)
plot_training_curves(training_logs, training_curve_path)
for path in [roc_figure_path, pr_figure_path, training_curve_path]:
    if not path.exists():
        raise FileNotFoundError(path)
print("ROC figure:", roc_figure_path)
print("PR figure:", pr_figure_path)
print("Training curves:", training_curve_path)


ROC figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\contrastive\roc_curves\contrastive_resnet18_ps32_roc_all_folds.png
PR figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\contrastive\pr_curves\contrastive_resnet18_ps32_pr_all_folds.png
Training curves: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\contrastive\training_curves\contrastive_resnet18_ps32_training_curves.png


## 14. Compare summary metrics against scratch and masked-recon baselines


In [13]:
summary_metrics = {}
for metric in [
    "auc", "pr_auc", "accuracy_05", "precision_05", "recall_05", "f1_05",
    "accuracy_best_f1", "precision_best_f1", "recall_best_f1", "f1_best_f1",
]:
    summary_metrics[f"mean_{metric}"] = float(fold_metrics[metric].mean())
    summary_metrics[f"std_{metric}"] = float(fold_metrics[metric].std(ddof=1))

worst_fold_index = int(fold_metrics["auc"].idxmin())
best_fold_index = int(fold_metrics["auc"].idxmax())
summary_metrics.update(
    {
        "model": model_name,
        "pretraining": pretraining,
        "patch_size": patch_size,
        "worst_fold_auc": float(fold_metrics.loc[worst_fold_index, "auc"]),
        "best_fold_auc": float(fold_metrics.loc[best_fold_index, "auc"]),
        "worst_fold": int(fold_metrics.loc[worst_fold_index, "fold"]),
        "best_fold": int(fold_metrics.loc[best_fold_index, "fold"]),
        "scratch_mean_auc": scratch_mean_auc,
        "scratch_std_auc": scratch_std_auc,
        "scratch_worst_fold_auc": scratch_worst_fold_auc,
        "masked_recon_mean_auc": masked_recon_mean_auc,
        "masked_recon_std_auc": masked_recon_std_auc,
        "masked_recon_worst_fold_auc": masked_recon_worst_fold_auc,
    }
)
summary_metrics["delta_mean_auc_vs_scratch"] = summary_metrics["mean_auc"] - scratch_mean_auc
summary_metrics["delta_std_auc_vs_scratch"] = summary_metrics["std_auc"] - scratch_std_auc
summary_metrics["delta_worst_auc_vs_scratch"] = summary_metrics["worst_fold_auc"] - scratch_worst_fold_auc
summary_metrics["delta_mean_auc_vs_masked_recon"] = summary_metrics["mean_auc"] - masked_recon_mean_auc
summary_metrics["delta_std_auc_vs_masked_recon"] = summary_metrics["std_auc"] - masked_recon_std_auc
summary_metrics["delta_worst_auc_vs_masked_recon"] = summary_metrics["worst_fold_auc"] - masked_recon_worst_fold_auc

summary_df = pd.DataFrame([summary_metrics])
summary_df.to_csv(summary_metrics_path, index=False)
print(summary_df.T.to_string(header=False))
print("Saved summary metrics:", summary_metrics_path)


mean_auc                                     0.838008
std_auc                                      0.084941
mean_pr_auc                                  0.831146
std_pr_auc                                   0.071061
mean_accuracy_05                             0.748769
std_accuracy_05                              0.062558
mean_precision_05                            0.790362
std_precision_05                             0.097063
mean_recall_05                               0.722677
std_recall_05                                0.234917
mean_f1_05                                   0.725748
std_f1_05                                    0.115135
mean_accuracy_best_f1                        0.708672
std_accuracy_best_f1                         0.068812
mean_precision_best_f1                         0.7579
std_precision_best_f1                        0.117523
mean_recall_best_f1                          0.712213
std_recall_best_f1                           0.328018
mean_f1_best_f1             

## 15. Print final summary and next-step notes


In [14]:
mean_std_table = pd.DataFrame(
    {
        "metric": ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"],
        "mean": [summary_metrics[f"mean_{m}"] for m in ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]],
        "std": [summary_metrics[f"std_{m}"] for m in ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]],
    }
)
print("Fold-wise metrics table:")
print(fold_metrics[["fold", "auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]].to_string(index=False))
print("\nMean +/- std metrics:")
print(mean_std_table.to_string(index=False))
print("\nWorst fold by AUC:", summary_metrics["worst_fold"], summary_metrics["worst_fold_auc"])
print("Best fold by AUC:", summary_metrics["best_fold"], summary_metrics["best_fold_auc"])
print("Scratch baseline mean_auc/std_auc:", scratch_mean_auc, scratch_std_auc)
print("Masked-recon mean_auc/std_auc:", masked_recon_mean_auc, masked_recon_std_auc)
print("Contrastive mean_auc/std_auc:", summary_metrics["mean_auc"], summary_metrics["std_auc"])
print("delta_mean_auc_vs_scratch:", summary_metrics["delta_mean_auc_vs_scratch"])
print("delta_std_auc_vs_scratch:", summary_metrics["delta_std_auc_vs_scratch"])
print("delta_worst_auc_vs_scratch:", summary_metrics["delta_worst_auc_vs_scratch"])
print("delta_mean_auc_vs_masked_recon:", summary_metrics["delta_mean_auc_vs_masked_recon"])
print("delta_std_auc_vs_masked_recon:", summary_metrics["delta_std_auc_vs_masked_recon"])
print("delta_worst_auc_vs_masked_recon:", summary_metrics["delta_worst_auc_vs_masked_recon"])
print("\nSaved output locations:")
print("predictions:", prediction_dir)
print("metrics:", metrics_dir)
print("training logs:", training_log_dir)
print("checkpoints:", checkpoint_dir)
print("figures:", figure_root)
print("\nImproved mean AUC relative to scratch:", summary_metrics["delta_mean_auc_vs_scratch"] > 0)
print("Improved worst-fold AUC relative to scratch:", summary_metrics["delta_worst_auc_vs_scratch"] > 0)
print("Improved fold-to-fold stability relative to scratch:", summary_metrics["delta_std_auc_vs_scratch"] < 0)
print("Mean AUC better than masked reconstruction:", summary_metrics["delta_mean_auc_vs_masked_recon"] > 0)
print("Worst-fold AUC better than masked reconstruction:", summary_metrics["delta_worst_auc_vs_masked_recon"] > 0)


Fold-wise metrics table:
 fold      auc   pr_auc  accuracy_05    f1_05  accuracy_best_f1  f1_best_f1
    0 0.872070 0.819683     0.812500 0.818182          0.781250    0.805556
    1 0.704142 0.714570     0.705128 0.646154          0.692308    0.600000
    2 0.910644 0.883535     0.820755 0.842975          0.773585    0.815385
    3 0.806122 0.848101     0.714286 0.750000          0.678571    0.742857
    4 0.897059 0.889842     0.691176 0.571429          0.617647    0.409091

Mean +/- std metrics:
          metric     mean      std
             auc 0.838008 0.084941
          pr_auc 0.831146 0.071061
     accuracy_05 0.748769 0.062558
           f1_05 0.725748 0.115135
accuracy_best_f1 0.708672 0.068812
      f1_best_f1 0.674578 0.171529

Worst fold by AUC: 1 0.7041420118343196
Best fold by AUC: 2 0.9106443574225703
Scratch baseline mean_auc/std_auc: 0.778842 0.104754
Masked-recon mean_auc/std_auc: 0.820912 0.043011
Contrastive mean_auc/std_auc: 0.8380075908531788 0.08494126273870213
